# Lesson 3: Runtime storage and predefined workers

In this example we extend the graph from the previous lesson to compile a circuit using `tkr-pytket-worker` and simulate it using `tkr-aer-worker`.

The Rust-backed runtime owns storage and execution. Worker implementation packages install commands such as `tkr-pytket-worker` and `tkr-aer-worker`; the default runtime discovers those commands on `PATH`.


In [ ]:
from typing import NamedTuple

from my_example_worker import substitute, symbolic_circuit
from tierkreis.builder import Graph
from tierkreis.controller.data.models import TKR


class PytketInputs(NamedTuple):
    a: TKR[float]
    b: TKR[float]
    c: TKR[float]


graph = Graph(PytketInputs, TKR[float])

circuit = graph.task(symbolic_circuit())
substituted = graph.task(
    substitute(circuit, graph.inputs.a, graph.inputs.a, graph.inputs.a)  # type: ignore
)

Now we add the compilation step.
For generic a generic the `tkr-pytket-worker` has `compile_generic_with_fixed_pass`.
You can find a list of all available tasks [here](../worker/native_workers/pytket_worker.md)

In [ ]:
from pytket_worker import add_measure_all, compile_generic_with_fixed_pass

compiled_circuit = graph.task(
    compile_generic_with_fixed_pass(circuit, optimisation_level=graph.const(2))
)
measured = graph.task(add_measure_all(compiled_circuit))  # type: ignore

Next we will run the circuit on `aer` with the `tkr-aer-worker`s `run_circuit` (see [docs](../worker/native_workers/aer_worker.md))
And calculate the expected value.
The `expectation` tasks simply assumes the computational basis.
For other observables you could defined another task in the `my-example-worker` as before.

In [ ]:
from aer_worker import run_circuit
from pytket_worker import expectation

results = graph.task(
    run_circuit(
        circuit=compiled_circuit,  # type: ignore
        n_shots=graph.const(10),
    ),
)
exp_val = graph.task(expectation(results))
workflow = graph.finish_with_outputs(exp_val)

## Creating a runtime

`new_default()` creates a runtime with file-backed assets, in-memory run state, and subprocess worker execution. Use `new_in_memory()` instead for workflows containing only built-in tasks.


In [ ]:
from shutil import which

from tierkreis import new_default

assert which("tkr-pytket-worker") is not None
assert which("tkr-aer-worker") is not None
runtime = await new_default()


The subprocess executor invokes installed worker commands directly. Source registry paths and executor objects are no longer passed for each run.


In [ ]:
# Worker commands are supplied by the installed implementation packages.


The runtime also creates workflow and run identifiers, so applications do not need to choose UUIDs or clean a checkpoint directory before rerunning a workflow.


In [ ]:
# A new call to start_new_run creates an independent run.


Save the workflow once, start a run with its inputs, wait for attempt `0`, and fetch the decoded outputs from the runtime.


In [ ]:
with runtime:
    workflow_id = await runtime.save_workflow("quantinuum_submission", workflow)
    run_id = await runtime.start_new_run(
        workflow_id, {"a": -1, "b": 0, "c": 1}
    )
    await runtime.wait_for(run_id, 0)
    outputs = await runtime.get_outputs(run_id, 0)

print(outputs)
